# Análisis de la Balanza de Pagos - BCRP

Fuente: [BCRP - Estadísticas](https://www.bcrp.gob.pe/estadisticas.html)

Este notebook carga y prepara la data de Balanza de Pagos para su análisis y visualización.

## 1. Configuración inicial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

print('Configuración lista ✅')

## 2. Carga de datos

El CSV tiene una estructura particular:
- **Fila 1**: vacía (se saltea)
- **Fila 2**: códigos de serie (PM39985BA, etc.)
- **Fila 3**: descripciones largas de cada serie
- **Filas 4+**: datos con el año en la primera columna
- **Encoding**: ISO-8859-1 (Latin-1), **no** UTF-8

In [ ]:
# Carga cruda: saltamos fila 1 (vacía), sin cabecera
df_raw = pd.read_csv(
    'assets/Balanza_de_Pagos.csv',
    encoding='ISO-8859-1',
    skiprows=1,
    header=None
)

print(f'Dimensiones raw: {df_raw.shape}')
df_raw.head(3)

### 2.1 Separar metadatos y datos

In [ ]:
# Códigos de serie (fila 0 del raw)
codigos = df_raw.iloc[0, 1:].tolist()

# Descripciones largas (fila 1 del raw)
descripciones = df_raw.iloc[1, 1:].tolist()

# Años disponibles
años = df_raw.iloc[2:, 0].tolist()

print(f'{len(codigos)} series cargadas')
print(f'Años: {años[0]} – {años[-2]} (2026 sin data)')
print(f'Códigos de ejemplo: {codigos[:3]}...')

### 2.2 Construir DataFrame limpio

In [ ]:
# Nombres cortos para trabajar más cómodo
nombres_cortos = [
    'CF_Sector_Privado',          # 1  PM39985BA
    'Efecto_Valuacion',           # 2  PM39998BA
    'Var_Saldo_RIN',              # 3  PM39997BA
    'Resultado_BP',               # 4  PM39996BA
    'Errores_Omisiones',          # 5  PM39995BA
    'Financ_Excepcional',         # 6  PM39994BA
    'CF_CortoPlazo_Pasivos',      # 7  PM39993BA
    'CF_CortoPlazo_Activos',      # 8  PM39992BA
    'CF_CortoPlazo',              # 9  PM39991BA
    'CF_SectorPublico_Pasivos',   # 10 PM39990BA
    'CF_SectorPublico_Activos',   # 11 PM39989BA
    'CF_SectorPublico',           # 12 PM39988BA
    'CF_SectorPrivado_Pasivos',   # 13 PM39987BA
    'CF_SectorPrivado_Activos',   # 14 PM39986BA
    'Cuenta_Corriente',           # 15 PM39972BA
    'Cuenta_Financiera',          # 16 PM39984BA
    'CC_Remesas',                 # 17 PM39983BA
    'CC_Ingreso_Secundario',      # 18 PM39982BA
    'CC_Ingreso_Primario_Pub',    # 19 PM39981BA
    'CC_Ingreso_Primario_Priv',   # 20 PM39980BA
    'CC_Ingreso_Primario',        # 21 PM39979BA
    'CC_Servicios_Import',        # 22 PM39978BA
    'CC_Servicios_Export',        # 23 PM39977BA
    'CC_Servicios',               # 24 PM39976BA
    'CC_Bienes_Import',           # 25 PM39975BA
    'CC_Bienes_Export',           # 26 PM39974BA
    'CC_Bienes',                  # 27 PM39973BA
]

df = (
    df_raw.iloc[2:, 1:]              # solo datos numéricos
    .set_axis(nombres_cortos, axis=1)  # nombres cortos como columnas
    .replace('n.d.', np.nan)           # 'n.d.' → NaN
    .apply(pd.to_numeric, errors='coerce')  # todo a numérico
)
df.index = pd.to_datetime(años, format='%Y').year  # año como índice
df.index.name = 'Año'

# Quitamos 2026 si está totalmente vacío
df = df.dropna(how='all')

print(f'DataFrame final: {df.shape[0]} años × {df.shape[1]} series')
df.head()

## 3. Diccionario de variables

Relación entre código BCRP, nombre corto y descripción completa.

In [ ]:
diccionario = pd.DataFrame({
    'Código': codigos,
    'Nombre_Corto': nombres_cortos,
    'Descripción': descripciones
})

diccionario

## 4. Estadísticas descriptivas

In [ ]:
df.describe().round(2)

In [ ]:
# Verificar valores nulos
nulos = df.isnull().sum()
nulos[nulos > 0]

---
## 5. Visualización

A partir de acá se pueden construir los gráficos deseados.

In [ ]:
# Ejemplo: evolución de Cuenta Corriente vs Cuenta Financiera
fig, ax = plt.subplots()

ax.plot(df.index, df['Cuenta_Corriente'], 'o-', label='Cuenta Corriente')
ax.plot(df.index, df['Cuenta_Financiera'], 's--', label='Cuenta Financiera')
ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
ax.set_title('Balanza de Pagos: Cuenta Corriente vs Cuenta Financiera')
ax.set_ylabel('millones US$')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Ejemplo: componentes de la Cuenta Corriente
componentes_cc = ['CC_Bienes', 'CC_Servicios', 'CC_Ingreso_Primario', 'CC_Ingreso_Secundario']

ax = df[componentes_cc].plot(kind='bar', figsize=(14, 6))
ax.set_title('Componentes de la Cuenta Corriente')
ax.set_ylabel('millones US$')
ax.axhline(0, color='gray', linewidth=0.5)
plt.tight_layout()
plt.show()